In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
# Load train and test
train_df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\train_encoded.csv")
test_df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\test_encoded.csv")

In [3]:
# Replace 'label' with your actual target column name if different
X_train = train_df.drop('label', axis=1)
y_train = train_df['label']

X_test = test_df.drop('label', axis=1)
y_test = test_df['label']

y_train = y_train.astype('category')
y_test = y_test.astype('category')

In [4]:
!pip install xgboost lightgbm

   ---------------------------------------- 0.0/149.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/149.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/149.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/149.9 MB 1.7 MB/s eta 0:01:29
   ---------------------------------------- 0.8/149.9 MB 1.2 MB/s eta 0:02:01
   ---------------------------------------- 1.0/149.9 MB 1.1 MB/s eta 0:02:19
   ---------------------------------------- 1.0/149.9 MB 1.1 MB/s eta 0:02:19
   ---------------------------------------- 1.0/149.9 MB 1.1 MB/s eta 0:02:19
   ---------------------------------------- 1.3/149.9 MB 799.2 kB/s eta 0:03:06
   ---------------------------------------- 1.6/149.9 MB 856.1 kB/s eta 0:02:54
    --------------------------------------- 2.1/149.9 MB 995.1 kB/s eta 0:02:29
    --------------------------------------- 2.4/149.9 MB 1.1 MB/s eta 0:02:18
    --------------------------------------- 2.6/149.9 MB 1.1 MB/s eta 0:02:11
 

In [6]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings("ignore")

# 1. Prepare Data
X_train = train_df.drop('label', axis=1)
y_train = train_df['label'].astype('category')

X_test = test_df.drop('label', axis=1)
y_test = test_df['label'].astype('category')

label_mapping = {-1: 0, 0: 1, 1: 2}

y_train = y_train.map(label_mapping)
y_test = y_test.map(label_mapping)

# 2. Models to Evaluate
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced'),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'),
    "LightGBM": LGBMClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM (RBF Kernel)": SVC(probability=True),
    "Naive Bayes": GaussianNB()
}

# 3. Cross Validation Setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 4. Evaluate all models using Accuracy and F1 Score
results = []
for name, model in models.items():
    pipeline = Pipeline([("scaler", StandardScaler()), ("clf", model)])

    acc_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
    f1_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1_macro')  # for multiclass

    results.append({
        "Model": name,
        "CV Accuracy": acc_scores.mean(),
        "CV F1 Macro": f1_scores.mean()
    })

# 5. Show results
results_df = pd.DataFrame(results).sort_values(by='CV F1 Macro', ascending=False)
print(results_df)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002389 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1762
[LightGBM] [Info] Number of data points in the train set: 8787, number of used features: 35
[LightGBM] [Info] Start training from score -1.428483
[LightGBM] [Info] Start training from score -1.673711
[LightGBM] [Info] Start training from score -0.557257
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002733 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1744
[LightGBM] [Info] Number of data points in the train set: 8787, number of used features: 35
[LightGBM] [Info] Start training from score -1.428483
[LightGBM] [Info] Start training from score -1.673711
[LightGBM] [Info] Start training from score -0.557257
[LightGBM] [Info] Auto-choosing col-